# Unit 4 — Supervised Fine-Tuning (SFT)

> *Part of the [RL for Robotics & LLMs](https://github.com/AliBuildsAI/rl-for-robotics-llms) series.*

Units 1–3 built policy gradients on robots, where the **environment gives a
reward**.  Units 4–8 move to language models, where it does not.  But before any
RL, there is a supervised step that everyone does first: **SFT**.

A *pretrained* language model knows grammar and facts, but it only does one
thing — predict the next token of internet text.  Ask it a question and it may
continue the question, or drift, rather than *answer*.  **Supervised
fine-tuning** teaches it to follow instructions by training on
(instruction, good-response) pairs.

This is the first hands-on stage of the RLHF pipeline, and the model we train
here is the **starting point for the rest of the series** — the reward model in
Unit 5 and the policy we optimise in Units 6–8 all build on an SFT model.

The training loop follows **Nathan Lambert's reference SFT code** from the RLHF
book ([`code/instruction_tuning`](https://github.com/natolambert/rlhf-book/tree/main/code/instruction_tuning)) —
a plain, inspectable PyTorch loop whose loss curve the book documents, so we know
the recipe trains correctly.

**What we build in this unit:**

| Part | What | Outcome |
|------|------|---------|
| A | Load a *base* LM + an instruction dataset | See the (prompt → response) format |
| B | A from-scratch SFT loop (mask prompt → CE on response → AdamW) | An instruction-following model |
| C | Before/after generation | Watch the base model start to *answer* instead of ramble |


---
## 1  Where SFT Sits — The Pipeline

A modern chat model is built in three stages:

| Stage | Trains on | Result |
|-------|-----------|--------|
| **1. Pretraining** | Trillions of internet tokens (next-token prediction) | Knows language & facts, but doesn't follow instructions |
| **2. SFT** *(this unit)* | (instruction, response) demonstrations | Follows instructions, as well as the demos |
| **3. RLHF** *(Units 5–8)* | Preferences / rewards | Goes *beyond* the demonstrations |

SFT is the bridge from a raw pretrained model to something that behaves like an
assistant.  It is also the **ceiling-setter for imitation**: the model can only
be as good as the responses it's shown.  That ceiling is exactly why RLHF
exists — to push past "imitate the demo" toward "produce what people actually
prefer."  But you need a competent instruction-follower first, and that's SFT.


---
## 2  What SFT Actually Is — Next-Token Prediction, Curated

Here is the part that surprises people: **SFT uses the exact same loss as
pretraining** — cross-entropy on the next token. Nothing about the objective is
new.  Two things change:

1. **The data.** Instead of random internet text, we train on curated
   (instruction → response) pairs, formatted with the model's **chat template**
   (the special tokens that mark *user* and *assistant* turns).

2. **What we compute the loss on.** We only want the model to learn to produce
   the **response**, not to re-generate the user's prompt.  So we **mask the
   prompt tokens out of the loss** and train only on the assistant's tokens.

That second point is the one genuinely SFT-specific idea, and it's a favourite
interview question. The next section makes it concrete.


### Chat templates — the format SFT trains on

How does a model know where the user stops and the assistant starts?  A **chat
template** wraps each turn in special tokens.  The de-facto standard is **ChatML**
(introduced by OpenAI), with three roles:

- **system** — standing instructions for the assistant; appears once, at the start
- **user** — the human's message
- **assistant** — the model's response

A formatted conversation looks like this in **ChatML** (what Qwen uses — OLMo
has its own markers, which we print later):

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
How many helicopters can a human eat in one sitting?<|im_end|>
<|im_start|>assistant
```

Two things to note (they hold for any template, whatever its exact markers):

- The trailing **assistant header** is the **generation prompt** — where the model
  begins writing. (`apply_chat_template(..., add_generation_prompt=True)` produces
  exactly this; we use it for inference.)
- Once *all* the SFT (and later preference) data is packed into one consistent
  template, models follow it with near-perfect reliability — which makes the
  end-of-turn special token a clean, learnable "stop" signal.

This template is also exactly what the loss-masking in the next section operates
on: everything up to and including the assistant header is *prompt* (masked); the
assistant's text plus its closing end-of-turn token is the *completion* (trained).


---
## 3  The Key Detail — Completion-Only Loss Masking

For a training example, we build one token sequence:

```
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
The capital of France is Paris.<|im_end|>
```

We feed the whole thing to the model, but for the **labels** we replace every
token belonging to the prompt (the system/user part, up to and including the
`assistant` header) with the ignore-index **`-100`**.  PyTorch's cross-entropy
skips any position whose label is `-100`, so **no gradient flows from the prompt
tokens** — the model is graded only on the response it should have produced.

Why it matters:

- If you *didn't* mask, the model would also be trained to generate user prompts
  — wasting capacity and teaching the wrong behaviour.
- The masked positions still serve as **context** (the model attends to them);
  they just don't contribute to the loss.

We build these masked labels **by hand** — first for one example in §B so you can
see the `-100`s with your own eyes, then in the real data pipeline (Part B, Step 1),
following Nathan's `_encode_row`.


---
## 4  Setup

A modern (Ampere-or-newer) GPU is assumed — A100, H100, L4, or an RTX 30/40/50
card (incl. the **4090**), all of which support **bf16**.  We train **OLMo-2-1B**
with fp32 master weights + bf16 autocast (batch 4, 2048-length, gradient
checkpointing), needing roughly **~18–24 GB** — comfortable on an A100 (40 GB) or
H100 (80 GB), and it fits a 24 GB card.


In [ ]:
%pip install -q "transformers>=4.45" "datasets>=3.0" accelerate matplotlib
print("All packages ready.")


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__}  |  device: {device}")
if device.type != "cuda":
    print("WARNING: no GPU detected — training will be slow. "
          "Runtime → Change runtime type → GPU.")


### Base model

We use **OLMo-2-0425-1B** — the *base* model, **not** the SFT/Instruct version.
That's the whole point: we do the instruction-tuning ourselves and watch it work.
This is **Nathan Lambert's exact model**, so the rest of his open code (reward
models, PPO, DPO, GRPO) drops straight into later units.

One wrinkle with base models: the tokenizer has **no chat template**, so we borrow
it from the instruction-tuned sibling (`allenai/OLMo-2-0425-1B-SFT`) — exactly what
Nathan's config does via `chat_template_source`.

`AutoModelForCausalLM` loads the model with its **language-modelling head** intact
(`hidden → vocab`) — unlike Unit 5's reward model, which swaps that head for a
scalar.  Here we keep predicting tokens.


#### The loading arguments

| Argument | What it is | Choice |
|---|---|---|
| `from_pretrained(MODEL_NAME)` | A Hub repo id (or local path); downloads config + weights + tokenizer. | `allenai/OLMo-2-0425-1B` — the **base** checkpoint |
| `tokenizer.chat_template = …` | The Jinja template that formats messages into text. | copied from the SFT variant (base has none) |
| `AutoModelForCausalLM` | Loads the model **with** its next-token (LM) head — a generative model. | the right class for SFT (we train it to generate) |
| `torch_dtype=torch.float32` | Precision the weights load in. | **fp32** for full fine-tuning (bf16 mixed precision is applied in the loop via autocast — see the note below) |
| `tokenizer.pad_token = eos_token` | Padding token for batching. | base tokenizers often have none → reuse EOS |


**Inputs → outputs of the two HuggingFace loaders:**

- `AutoTokenizer.from_pretrained(model_id)` — **in:** a Hub repo id (string).
  **out:** a *tokenizer* object (text ↔ token-id converter + the chat template).
- `AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=...)` — **in:** the
  repo id + load precision.  **out:** the *model* (an `nn.Module` with the
  language-modelling head).

> **Load in fp32 for full fine-tuning** (the one place we differ from Nathan, who
> loads bf16).  We pass `torch_dtype=torch.float32`.  The forward/backward still
> run fast in **bf16 via autocast** in the training loop, but the **master weights
> stay fp32** so small optimizer updates (`lr × grad`) actually accumulate.
> Loading the weights directly in bf16 (~3 significant figures) makes those tiny
> updates round away — the loss barely moves.  (This is about correctness, not GPU
> compatibility.)


In [ ]:
MODEL_NAME           = "allenai/OLMo-2-0425-1B"       # Nathan's exact base model
CHAT_TEMPLATE_SOURCE = "allenai/OLMo-2-0425-1B-SFT"   # borrow the chat template from the SFT variant

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# The base model's tokenizer has no chat template. Take it from the instruction-tuned
# variant — exactly what Nathan's config does via `chat_template_source`.
tokenizer.chat_template = AutoTokenizer.from_pretrained(CHAT_TEMPLATE_SOURCE).chat_template
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# fp32 weights → bf16 mixed precision is applied via autocast in the training loop,
# keeping fp32 master weights so gradient updates don't underflow.
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_NAME}  |  {n_params/1e6:.0f}M params")
print("chat template set:", tokenizer.chat_template is not None)


---
## Part A — The Instruction Data

We use **No Robots** (`HuggingFaceH4/no_robots`) — the same ~9.5K
hand-written instruction dataset Nathan Lambert's reference SFT code uses, and
the one the chapter calls out as an early state-of-the-art set.  Each row has a
`messages` column: a list of `{"role": ..., "content": ...}` turns — the chat
format we tokenise with `apply_chat_template`.


#### `load_dataset` arguments

| Argument | What it is | Choice |
|---|---|---|
| `load_dataset("HuggingFaceH4/no_robots")` | Dataset repo id on the Hub. | conversational, already in `messages` format; matches Nathan's SFT recipe |
| `["train"]` | Pick the split. | the full ~9.5K training split (small enough to use whole) |

**`load_dataset(repo_id)` — in:** a dataset repo id (string).  **out:** a
`DatasetDict` — a dict of splits (`"train"`, `"test"`), each a `Dataset` (a
table of rows).  Indexing a split with `[i]` returns one row as a dict.


In [ ]:
dataset = load_dataset("HuggingFaceH4/no_robots")

train_ds = dataset["train"]          # full ~9.5K rows (no subsampling — matches Nathan's recipe)
eval_ds  = dataset["test"]           # held-out ~500 rows — used to measure eval loss
print(f"Train examples: {len(train_ds)}")
print(f"Eval  examples: {len(eval_ds)}")
print(f"Columns: {train_ds.column_names}")


**`tokenizer.apply_chat_template(messages, tokenize=False)` — in:** a list of
`{"role", "content"}` messages + flags.  **out:** with `tokenize=False`, the
**formatted string** with the model's special role-marker tokens inserted; with
`tokenize=True` it returns **token ids** instead.  `add_generation_prompt=True`
appends the trailing `<|im_start|>assistant\n` so the model knows to start replying.


In [ ]:
# Inspect one example: the raw messages, then what the chat template produces.
ex = train_ds[0]
print("=== messages ===")
for m in ex["messages"]:
    print(f"[{m['role']}] {m['content'][:200]}")

print("\n=== after apply_chat_template (what the model actually sees) ===")
print(tokenizer.apply_chat_template(ex["messages"], tokenize=False)[:600])


### How much data? — quality over quantity

A natural question: how many examples do you need?  The history is instructive:

- **~10K hand-written** examples (e.g. the *No Robots* dataset) were
  state-of-the-art early on.
- **LIMA** ("Less Is More for Alignment") showed that as few as **~1,000**
  carefully curated examples can produce a strong instruction-follower — evidence
  that SFT mostly *surfaces* abilities already in the pretrained model rather than
  teaching new ones.
- Today, **large synthetic datasets** (generated by stronger models) work best on
  most tasks.  Around **~1M prompts** is enough for excellent post-training;
  beyond that, **returns diminish quickly**.

Two principles that matter more than raw count:

1. **Quality > quantity.** A small, clean, high-quality set beats a large noisy one.
2. **Distribution match.** The best prompts look like the **downstream tasks you
   care about** — SFT data should resemble what users will actually ask.

There's also a useful robustness fact: *if more stages follow SFT* (reward
modelling, RL), the model can **recover from some noise** in the SFT data — so
SFT doesn't have to be perfect, it has to be a good starting point.  No Robots is
small enough (~9.5K) that we train on the **whole thing**, exactly as Nathan's
reference recipe does.


---
## Part B — Before vs After

First, a helper to generate an answer, and a look at how the **base** model
responds *before* any SFT.  Base models often continue or ramble rather than give
a clean, bounded answer.

**The HuggingFace calls in `generate`, in order (in → out):**

- `tokenizer(text, return_tensors="pt")` — **in:** a string.  **out:** a dict of
  tensors — `input_ids` (token ids, shape `(1, seq_len)`) and `attention_mask`.
- `model.generate(**inputs, max_new_tokens=…, do_sample=False)` — **in:** the
  input ids + decoding settings.  **out:** a tensor of ids `(1, seq_len + new)`
  = the prompt followed by the generated continuation.  `do_sample=False` = greedy.
- `tokenizer.decode(ids, skip_special_tokens=True)` — **in:** a list/tensor of
  token ids.  **out:** the text string (special tokens stripped).  We slice off
  the prompt ids first so we return only the *new* tokens.


In [ ]:
@torch.no_grad()
def generate(prompt, model, max_new_tokens=128):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.pad_token_id)
    # decode only the newly generated tokens (skip the prompt)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:],
                            skip_special_tokens=True)

PROMPTS = [
    "Give me three tips for writing clean code.",
    "Explain what a binary search is in one sentence.",
]

print("=== BASE MODEL (before SFT) ===")
for p in PROMPTS:
    print(f"\nQ: {p}\nA: {generate(p, model)}")


### Token anatomy of a chat prompt

A quick look at how the chat template tokenises.  Every chat model wraps each turn
in **special role-marker tokens** (Qwen uses `<|im_start|>`/`<|im_end|>`; OLMo uses
its own).  These markers are single tokens added to the vocabulary; the role names
and the message text are *ordinary* tokens.  The cell prints your model's actual
template + its special tokens — run it to see what OLMo uses.


In [ ]:
# whole formatted prompt — print it to see this model's special role markers
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": "How many helicopters can a human eat in one sitting?"}],
    tokenize=False, add_generation_prompt=True)
print("=== formatted prompt ===")
print(text)

ids = tokenizer(text, add_special_tokens=False)["input_ids"]
print("\ntotal tokens:", len(ids))
print("special tokens this template uses:", tokenizer.all_special_tokens)


### See the loss mask (the `-100`s)

Before training, let's build the masked labels for one example **by hand** —
the thing §3 described.  We tokenise the full conversation, then the prompt
portion alone, and set the first `len(prompt)` labels to `-100`.


In [ ]:
demo = [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
]

full_text   = tokenizer.apply_chat_template(demo, tokenize=False)
prompt_text = tokenizer.apply_chat_template(demo[:1], tokenize=False,
                                            add_generation_prompt=True)

full_ids   = tokenizer(full_text,   add_special_tokens=False)["input_ids"]
prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]

# completion-only labels: -100 over the prompt, real token ids over the response
labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

print(f"{'token':>14} | {'label':>6}")
print("-" * 26)
for tok_id, lab in zip(full_ids, labels):
    tok = tokenizer.decode([tok_id]).replace("\n", "\\n")
    shown = "-100 (masked)" if lab == -100 else str(lab)
    print(f"{tok!r:>14} | {shown:>6}")

masked = sum(l == -100 for l in labels)
print(f"\n{masked}/{len(labels)} tokens masked (the prompt). "
      f"Loss is computed only on the {len(labels)-masked} response tokens.")


### What about multi-turn conversations?

Our demo had one user turn and one assistant turn.  Real conversations have many.
There are two standard masking choices:

1. **Final-turn only** — mask everything except the *last* assistant turn. Simple,
   but you waste the earlier assistant turns as training signal.
2. **Mask user turns only** — mask every user turn, but include **all** assistant
   turns in the loss.  This is the common choice: the model learns from every
   response it should have produced, while never being trained to generate the
   user's messages.

In both cases the rule is the same one idea as before — **never compute loss on a
user/prompt token**.  Our `encode_row` below does the *final-turn-only* version
(mask everything except the last assistant turn); it's worth knowing both options
exist because interviewers ask "how do you handle multi-turn data?"


## Part B — Training: a from-scratch SFT loop (Nathan Lambert's recipe)

Instead of a `Trainer` abstraction, we follow **Nathan Lambert's reference SFT
code** (`code/instruction_tuning` in the RLHF-book repo) — a plain PyTorch loop.
The book documents this exact recipe's loss curve, so we know it trains
correctly, and every piece is explicit: **mask the prompt → cross-entropy on the
response → AdamW with warmup + linear decay → sample generations to watch it
learn.**  No hidden magic.


### Choosing the hyperparameters — what real recipes use

The single most important SFT hyperparameter is the **learning rate**, and the
rule of thumb is sharp: **SFT uses an LR one to two orders of magnitude smaller
than pretraining.** You are *adjusting* a model that already works, not building
one — too high an LR wipes out pretrained knowledge.  Real open recipes:

| Model | Pretraining LR | SFT LR |
|---|---|---|
| OLMo 2 | $3\times10^{-4}$ | $1\times10^{-5}$ |
| OLMo 3 | — | $5\text{–}8\times10^{-5}$ |

We use Nathan's exact **`5e-6`** for OLMo-2-1B — the value his documented loss
curve was produced with.  A few more practitioner notes from the book:

- **Schedule:** warm up over a small fraction of steps, then decay (often
  linearly) to near zero.
- **Batch size** is much smaller than pretraining (e.g. OLMo 2 used ~256 prompts
  per step vs 1–2K rows in pretraining).  The *linear scaling rule*: bigger
  batches give lower-variance gradients, which *supports* a higher LR.
- **Epochs:** 1–3 is typical.  More risks memorising the demos.
- **In practice teams sweep a few LRs and pick the best checkpoint on a held-out
  eval** — there's no single magic number.

The meta-point worth saying on camera: *crafting the overall pipeline matters more
than perfecting any one stage.* SFT just needs to be a solid starting point for
the RLHF that follows.


### How do we know SFT worked? (and why NOT to trust val loss)

Your instinct is probably "watch the held-out loss go down." For SFT that
instinct is **wrong**, and it's worth understanding why.

- **Train loss should drop** — it confirms the model is actually learning the
  data (if it stays flat, something is broken, e.g. weights loaded in bf16).
- **Val (held-out) loss often *rises*** during SFT.  Two reasons: (1) mild
  overfitting on a small dataset, and (2) more fundamentally, SFT **shifts the
  model's output distribution** toward the assistant style — which makes it *more*
  surprised by the held-out reference text, so cross-entropy on it goes **up**
  even as the model becomes a better instruction-follower.

So **SFT is judged behaviourally**, by the responses it produces — not by
perplexity.  (This is exactly why Nathan's reference config samples *generations*
during training instead of watching val loss, and why real pipelines use
instruction benchmarks like IFEval / MT-Bench.)  We still *report* val loss each
epoch — but to *observe* this rise, not to chase it.  We keep the **final** model,
and the real proof is the before/after generations in Part C.


### Step 1 — encode each example with prompt masking

Nathan's masking (`_encode_row`), done by hand: tokenise the **prompt** (all
messages except the final assistant turn, with the generation prompt) and the
**full** conversation, then set the prompt positions to `-100` so only the
response contributes to the loss.

- `tokenizer.apply_chat_template(msgs[:-1], tokenize=True, add_generation_prompt=True)`
  — **in:** the messages up to the reply + flags.  **out:** the *prompt token ids*.
- `apply_chat_template(msgs, tokenize=True)` — **out:** the *full token ids*.
- `labels = [-100]*len(prompt) + full[len(prompt):]` — mask the prompt, keep the response.


In [ ]:
IGNORE_INDEX = -100
MAX_LENGTH   = 2048

def encode_row(example):
    msgs = example["messages"]
    prompt_ids = tokenizer.apply_chat_template(
        msgs[:-1], tokenize=True, add_generation_prompt=True)      # prompt only
    full_ids = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=False)          # prompt + response
    labels = [IGNORE_INDEX] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {"input_ids": full_ids[:MAX_LENGTH], "labels": labels[:MAX_LENGTH]}

train_enc = train_ds.map(encode_row, remove_columns=train_ds.column_names)
eval_enc  = eval_ds.map(encode_row,  remove_columns=eval_ds.column_names)

ex = train_enc[0]
masked = sum(l == IGNORE_INDEX for l in ex["labels"])
print(f"example: {len(ex['input_ids'])} tokens  |  {masked} prompt tokens masked, "
      f"{len(ex['labels']) - masked} response tokens trained")


### Step 2 — batch with a padding collate

A `DataLoader` groups examples into batches.  Our `collate` pads every sequence
in a batch to the same length: `input_ids` with the pad token, `attention_mask`
with 0 over the padding, and **`labels` with `-100`** over the padding (so pad
tokens never contribute to the loss, same as the prompt).

- `DataLoader(dataset, batch_size, shuffle, collate_fn)` — **in:** the encoded
  dataset + how to batch.  **out:** an iterable yielding batched tensor dicts.


In [ ]:
from torch.utils.data import DataLoader

PAD_ID = tokenizer.pad_token_id

def collate(batch):
    maxlen = max(len(x["input_ids"]) for x in batch)
    input_ids, labels, attn = [], [], []
    for x in batch:
        n, pad = len(x["input_ids"]), maxlen - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [PAD_ID] * pad)
        labels.append(   x["labels"]    + [IGNORE_INDEX] * pad)
        attn.append(     [1] * n + [0] * pad)
    return {"input_ids":      torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn,      dtype=torch.long),
            "labels":         torch.tensor(labels,    dtype=torch.long)}

BATCH_SIZE = 4
train_loader = DataLoader(train_enc, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate)
eval_loader  = DataLoader(eval_enc,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
print(f"{len(train_loader)} batches/epoch")


### Step 3 — optimiser, schedule, loss, and a sampling probe

Nathan's recipe (LR sized for our 0.5B):

| Knob | Value | Note |
|---|---|---|
| optimiser | **AdamW**, `lr=5e-6`, `weight_decay=0.0` | Nathan's exact OLMo-1B value |
| schedule | linear **warmup 10%** then linear decay to 0 | `LambdaLR` |
| effective batch | `BATCH_SIZE(4) × ACCUM(8) = 32` | gradient accumulation |
| epochs | 3 | ~891 optimiser steps total |
| grad clip | `max_grad_norm=1.0` | |
| precision | **fp32 weights + bf16 autocast** | fp32 master weights (validated) |
| `SAMPLE_EVERY` | 50 steps | generate to *watch* it learn (his idea) |

`compute_loss` is the by-hand causal-LM loss: shift logits/labels by one and run
`cross_entropy` with `ignore_index=-100` (prompt + padding skipped).


In [ ]:
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

LR, EPOCHS, ACCUM = 5e-6, 3, 8       # Nathan's OLMo-1B recipe; effective batch = 4 × 8 = 32
WARMUP_RATIO, WEIGHT_DECAY, MAX_GRAD_NORM = 0.1, 0.0, 1.0
SAMPLE_EVERY = 50
torch.manual_seed(42)

# gradient checkpointing to fit 2048-length; cache must be off while training
model.gradient_checkpointing_enable()
model.config.use_cache = False

total_opt_steps = (len(train_loader) // ACCUM) * EPOCHS
warmup_steps    = int(total_opt_steps * WARMUP_RATIO)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(step):                                  # linear warmup, then linear decay to 0
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    return max(0.0, (total_opt_steps - step) / max(1, total_opt_steps - warmup_steps))
scheduler = LambdaLR(optimizer, lr_lambda)

# bf16 autocast for the forward/backward (fast on any modern GPU); master weights
# stay fp32 so updates accumulate.
USE_AUTOCAST = (device.type == "cuda")

def compute_loss(batch):
    logits = model(input_ids=batch["input_ids"],
                   attention_mask=batch["attention_mask"]).logits
    shift_logits = logits[:, :-1, :].contiguous().float()   # predict token t+1 from t
    shift_labels = batch["labels"][:, 1:].contiguous()
    return F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)),
                           shift_labels.view(-1), ignore_index=IGNORE_INDEX)

@torch.no_grad()
def sample(prompt, max_new_tokens=80):
    model.eval(); model.config.use_cache = True
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    inp = tokenizer(text, return_tensors="pt").to(device)
    out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.7, top_p=0.9, pad_token_id=PAD_ID)
    model.config.use_cache = False; model.train()
    return tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

print(f"total optimiser steps: {total_opt_steps}  (warmup {warmup_steps})")


### Step 4 — the training loop

The standard mixed-precision loop with gradient accumulation: forward under
`autocast`, scale the loss by `1/ACCUM`, `backward()`, and every `ACCUM`
micro-batches clip the gradients and take one optimiser + scheduler step.  Every
`SAMPLE_EVERY` steps we generate from a fixed prompt — you should watch it go
from **rambling → a single clean, terminated answer** (the book's step-100 →
step-650 story).


In [ ]:
MONITOR_PROMPT = "Give me three tips for writing clean code."
loss_hist, step_hist = [], []
opt_step = 0
model.train(); optimizer.zero_grad()

for epoch in range(EPOCHS):
    for i, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16,
                            enabled=USE_AUTOCAST):
            loss = compute_loss(batch)
        (loss / ACCUM).backward()

        if (i + 1) % ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
            opt_step += 1

            if opt_step % 25 == 0:
                loss_hist.append(loss.item()); step_hist.append(opt_step)
                print(f"epoch {epoch+1}  step {opt_step:4d}/{total_opt_steps}  "
                      f"loss {loss.item():.3f}  lr {scheduler.get_last_lr()[0]:.2e}")

            if SAMPLE_EVERY and opt_step % SAMPLE_EVERY == 0:
                print(f"   ↳ sample @ {opt_step}: {sample(MONITOR_PROMPT)[:160]!r}")

model.config.use_cache = True   # re-enable KV cache so Part C generation is fast
print("\nSFT complete.")


### The loss curve + held-out loss

Plot the training loss — you should see the **sharp early drop** (~first 100–150
steps, as the model learns the chat template) then a gradual decline, exactly as
the book describes.  We also report held-out loss once — remember it can *rise*;
it's not the success metric.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(step_hist, loss_hist)
plt.xlabel("optimiser step"); plt.ylabel("training loss")
plt.title("SFT training loss — sharp early drop, then gradual refinement")
plt.grid(alpha=0.3); plt.tight_layout(); plt.savefig("sft_loss.png", dpi=130); plt.show()

@torch.no_grad()
def held_out_loss():
    model.eval()
    tot = n = 0
    for batch in eval_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16,
                            enabled=USE_AUTOCAST):
            tot += compute_loss(batch).item(); n += 1
    model.train()
    return tot / n

print(f"train loss: {loss_hist[0]:.3f} → {loss_hist[-1]:.3f}   (should DROP — it's learning)")
print(f"held-out loss: {held_out_loss():.3f}   (may be flat/up — judge by generations, not this)")


---
## Part C — After SFT

Same prompts, same `generate` function — now on the fine-tuned model.  Look for:
the answer is **on-topic**, **bounded** (it stops cleanly at the end-of-turn token instead
of running on), and actually *follows the instruction* rather than continuing it.


In [ ]:
print("=== SFT MODEL (after) ===")
for p in PROMPTS:
    print(f"\nQ: {p}\nA: {generate(p, model)}")


### What to expect — read the two signals correctly

1. **Train loss dropped** (≈ 2.2 → ~1.5) — the model *is* learning. This is the
   signal that training is working at all.
2. **Held-out loss may have risen** — and that's fine (see the note above): SFT
   shifts the output distribution, so reference perplexity goes up even as
   behaviour improves. **Don't read the rising val loss as failure.**
3. **The real proof is right here — the generations.** The SFT model should be
   **on-topic, instruction-shaped, and properly terminated** (stops at its
   end-of-turn token), where the base model rambled, repeated, or didn't stop.

On a 0.5B model the qualitative change is real but not dramatic — but it should be
visibly *more* like an assistant than the base model above.

The honest framing for the video: SFT doesn't make the model *smarter* — the
knowledge was already in the pretrained weights.  It makes the model **use that
knowledge in the format we want** (answer the question, then stop).  Pushing
quality *beyond* the demonstrations is what Units 5–8 are for.


### Key takeaways

The whole unit in seven lines — the things worth saying out loud:

1. **SFT = the same next-token cross-entropy loss as pretraining**, on curated
   (instruction, response) data formatted with a chat template.
2. The one SFT-specific idea is **completion-only loss masking**: prompt tokens get
   label `-100` and contribute *no* gradient — the model is graded on the response.
3. **Chat templates** (ChatML: system / user / assistant) give the model a learnable
   structure and a clean end-of-turn stop token.
4. **Quality > quantity**: LIMA showed ~1K good examples go a long way; SFT mostly
   *surfaces* pretrained ability, it doesn't add knowledge.
5. **Learning rate is 1–2 orders of magnitude below pretraining** (~`5e-6`–`8e-5`;
   we use Nathan's `5e-6`); too high erases what the model already knows.
6. **SFT need not be perfect** — later RLHF stages recover from some noise; what
   matters is the *overall* pipeline, not any single stage.
7. SFT sets the **imitation ceiling**; Units 5–8 break past it with preferences and RL.


### Save the checkpoint — this feeds Unit 5

This SFT model is the artifact the rest of the series builds on.  Save it; in
Unit 5 you can point the reward model's `MODEL_NAME` at this folder instead of
the official Instruct checkpoint.

**Inputs → outputs:**

- `model.save_pretrained(dir)` — **in:** a folder path.  **out:** none; **writes**
  the model weights + config to disk (a folder you can later `from_pretrained`).
- `tokenizer.save_pretrained(dir)` — **in:** the same folder.  **out:** none;
  **writes** the tokenizer files alongside the model so the folder is self-contained.


In [ ]:
SAVE_DIR = "olmo-sft-final"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved SFT model + tokenizer → {SAVE_DIR}/")
print("In Unit 5: MODEL_NAME = \"" + SAVE_DIR + "\"  (or keep the official Instruct model)")


---
## 5  What's Next — Reward Models (Unit 5)

We now have an instruction-following model — but it only knows how to *imitate*
the responses it was shown.  It has no concept of one answer being **better** than
another.

That's the gap **Unit 5** fills.  We collect human **preference** comparisons
(answer A is better than answer B) and train a **reward model** $r_\phi(x, y)$
that scores any response.  Then in Units 6–8 we use RL — PPO, and later DPO and
GRPO — to push this SFT model *beyond* imitation toward responses people actually
prefer.

→ [Back to the series](https://github.com/AliBuildsAI/rl-for-robotics-llms)
